# External Validation — Prototype CNN on Two Unseen Datasets

Loads our trained 256×256 grayscale CNN (from `prototype-models/prototype_model.ipynb`) and evaluates it
on two **external** datasets it has never seen, to test generalization beyond the original training distribution.

**Datasets (via Kaggle API):**
1. **BRISC 2025** — `briscdataset/brisc2025`
2. **Brain Tumor MRI Dataset for Deep Learning** — `alamshihab075/brain-tumor-mri-dataset-for-deep-learning`

**Reported per dataset:** test accuracy, per-class recall (glioma / meningioma / notumor / pituitary), and a confusion matrix.

**Key correctness safeguards** (improving on the earlier external-eval notebook):
- An explicit, verified **folder-name → canonical-class mapping** (never trust `ImageFolder`'s alphabetical order blindly).
- The **exact eval transform used in training**, including the brain-crop step.
- `strict` weight loading so an architecture mismatch fails loudly.

## 1. Setup

In [ ]:
# On Colab, torch / opencv / matplotlib are pre-installed; add the extras only.
%pip install -q kaggle scikit-learn

In [ ]:
import os
import numpy as np
import cv2
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

In [ ]:
SEED = 42
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

IMG_SIZE = 256
BATCH_SIZE = 32
NORM_MEAN, NORM_STD = (0.5,), (0.5,)

# Output-index order the model was trained on (torchvision ImageFolder sorts Training/ alphabetically):
#   0=glioma, 1=meningioma, 2=notumor, 3=pituitary
CANONICAL_CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]

print("Using device:", DEVICE)

## 2. Download the unseen datasets (Kaggle API)

On Colab, add `KAGGLE_USERNAME` and `KAGGLE_KEY` in the Secrets panel (key icon) with notebook access enabled.

In [ ]:
# --- Kaggle credentials ---
try:
    from google.colab import userdata  # present only on Colab
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Loaded Kaggle credentials from Colab secrets.")
except ImportError:
    from dotenv import load_dotenv  # running locally
    load_dotenv(dotenv_path=os.path.join("..", ".env"))
    load_dotenv()
    print("Loaded Kaggle credentials from .env.")

In [ ]:
import kaggle

# name -> (kaggle slug, local folder)
DATASETS = {
    "BRISC 2025": ("briscdataset/brisc2025", "other1_brisc2025"),
    "Deep-Learning MRI": ("alamshihab075/brain-tumor-mri-dataset-for-deep-learning", "other2_deeplearning"),
}

kaggle.api.authenticate()
for name, (slug, path) in DATASETS.items():
    if os.path.isdir(path) and os.listdir(path):
        print(f"{name}: already present at {path}/ - skipping.")
    else:
        print(f"{name}: downloading {slug} ...")
        kaggle.api.dataset_download_files(slug, path=path, unzip=True)
        print(f"{name}: done -> {path}/")

## 3. Locate the class folders and build a verified class mapping

The external datasets may nest their class folders under a split (e.g. `test/`) or a task subfolder, and
may name classes differently (`no_tumor` vs `notumor`, etc.). We inspect the tree, auto-locate the folder
that holds the class subdirectories, and build an **explicit** mapping to our canonical class order —
failing loudly on any folder we can't recognize.

In [ ]:
def print_tree(root, max_depth=2, prefix=""):
    """Print the directory structure (folders only) up to max_depth."""
    if max_depth < 0 or not os.path.isdir(root):
        return
    for e in sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))):
        sub = os.path.join(root, e)
        n_files = sum(len(f) for _, _, f in os.walk(sub))
        print(f"{prefix}{e}/  ({n_files} files)")
        print_tree(sub, max_depth - 1, prefix + "    ")


def normalize(name):
    """Lowercase, strip separators, drop a trailing 'tumor' (but keep 'notumor')."""
    s = name.lower().strip().replace("_", "").replace("-", "").replace(" ", "")
    if s.endswith("tumor") and s != "notumor":
        s = s[:-5]
    return s


# Folder-name aliases -> canonical class. Extend this if a dataset uses an unusual name.
ALIASES = {
    "glioma": "glioma",
    "meningioma": "meningioma",
    "pituitary": "pituitary",
    "notumor": "notumor",
    "no": "notumor",       # "no_tumor" -> "no" after dropping trailing "tumor"
    "healthy": "notumor",
    "normal": "notumor",
}


def resolve_class(folder_name):
    """Map a dataset folder name to a canonical class, or None if unrecognized."""
    return ALIASES.get(normalize(folder_name))


def find_class_dir(root):
    """Return the directory whose immediate subfolders best match the 4 canonical classes,
    preferring a test/val split when several candidates exist."""
    candidates = []  # (num_matched_classes, prefers_test, path)
    for dirpath, dirnames, _ in os.walk(root):
        matched = {resolve_class(d) for d in dirnames}
        matched.discard(None)
        if len(matched) >= 2:
            is_test = any(t in dirpath.lower() for t in ("test", "val"))
            candidates.append((len(matched), is_test, dirpath))
    if not candidates:
        raise FileNotFoundError(f"No class-like folder found under {root}")
    candidates.sort(key=lambda c: (c[0], c[1]), reverse=True)
    return candidates[0][2]

In [ ]:
# Inspect both datasets' structure.
for name, (_, path) in DATASETS.items():
    print(f"\n===== {name}  ({path}) =====")
    print_tree(path, max_depth=2)

In [ ]:
# Auto-resolve the class directory for each dataset.
# If the tree printout above shows a better path, override these manually.
OTHER1_CLASS_DIR = find_class_dir("other1_brisc2025")
OTHER2_CLASS_DIR = find_class_dir("other2_deeplearning")
print("BRISC 2025 class dir      :", OTHER1_CLASS_DIR)
print("Deep-Learning MRI class dir:", OTHER2_CLASS_DIR)

In [ ]:
def build_remap(class_dir):
    """Map ImageFolder's (alphabetical) indices to canonical class indices. Fails on any
    unrecognized folder; warns on any canonical class absent from the dataset."""
    folders = sorted(d for d in os.listdir(class_dir) if os.path.isdir(os.path.join(class_dir, d)))
    remap = {}
    print(f"Class mapping for {class_dir}:")
    for imagefolder_idx, folder in enumerate(folders):  # ImageFolder assigns idx by sorted order
        canonical = resolve_class(folder)
        if canonical is None:
            raise ValueError(
                f"Folder '{folder}' in {class_dir} maps to no canonical class "
                f"{CANONICAL_CLASSES}. Add an entry to ALIASES."
            )
        canon_idx = CANONICAL_CLASSES.index(canonical)
        remap[imagefolder_idx] = canon_idx
        print(f"  '{folder}' (idx {imagefolder_idx}) -> {canonical} (canonical idx {canon_idx})")
    missing = set(CANONICAL_CLASSES) - {CANONICAL_CLASSES[v] for v in remap.values()}
    if missing:
        print(f"  WARNING: classes absent from this dataset: {sorted(missing)}")
    return remap

## 4. Rebuild the model and load the trained weights

The `CNN` class below is **identical** to the training notebook (256px, 1 channel, BatchNorm + dropout).
It must match, or the state_dict will not load. Point `WEIGHTS_PATH` at your fresh 256px `best_cnn.pt`.

In [ ]:
class CNN(nn.Module):
    def __init__(self, num_classes=4, in_channels=1):
        super().__init__()

        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(),
                nn.MaxPool2d(2),
            )

        self.conv = nn.Sequential(
            block(in_channels, 32),   # 256 -> 128
            block(32, 64),            # 128 -> 64
            block(64, 128),           # 64  -> 32
            block(128, 128),          # 32  -> 16
        )
        with torch.no_grad():
            n_flat = self.conv(torch.zeros(1, in_channels, IMG_SIZE, IMG_SIZE)).flatten(1).shape[1]
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(n_flat, 128), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.fc(self.conv(x))

In [ ]:
# --- Load trained weights ---
# Option A (default): mount Google Drive and point WEIGHTS_PATH at your best_cnn.pt there.
# Option B: upload the file directly (see the commented lines below).
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WEIGHTS_PATH = "/content/drive/MyDrive/best_cnn.pt"   # <-- edit to your file's location
except ImportError:
    WEIGHTS_PATH = "best_cnn.pt"   # local run

# Option B (upload instead of Drive):
# from google.colab import files
# WEIGHTS_PATH = next(iter(files.upload()))

model = CNN(num_classes=len(CANONICAL_CLASSES)).to(DEVICE)
# strict=True (default) raises if the architecture doesn't match the checkpoint.
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
model.eval()
print("Loaded weights from:", WEIGHTS_PATH)

## 5. Preprocessing (must match training exactly)

In [ ]:
# Same crop used at training time (crop away black margins around the brain).
def crop_brain_region(pil_img):
    gray = np.array(pil_img.convert("L"))
    thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)[1]
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return pil_img
    x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
    return pil_img.crop((x, y, x + w, y + h))


# Identical to the training notebook's eval_transform (NO augmentation).
eval_transform = transforms.Compose([
    transforms.Lambda(crop_brain_region),
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

## 6. Evaluation + metrics

In [ ]:
def make_loader(class_dir):
    """ImageFolder over class_dir with labels remapped into canonical index space."""
    remap = build_remap(class_dir)
    ds = datasets.ImageFolder(class_dir, transform=eval_transform)
    ds.target_transform = lambda i: remap[i]   # labels come out as canonical indices
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


def evaluate(model, loader, device):
    """Run the model over a loader; return (preds, targets) as canonical-index arrays."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            preds.extend(model(imgs).argmax(1).cpu().tolist())
            targets.extend(labels.tolist())
    return np.array(preds), np.array(targets)

In [ ]:
LABELS = list(range(len(CANONICAL_CLASSES)))  # [0, 1, 2, 3] -> keeps all classes present


def report(name, preds, targets):
    acc = accuracy_score(targets, preds)
    recalls = recall_score(targets, preds, labels=LABELS, average=None, zero_division=0)

    print(f"\n================  {name}  ================")
    print(f"Test accuracy: {acc:.4f}\n")
    print("Per-class recall:")
    for cls, r in zip(CANONICAL_CLASSES, recalls):
        print(f"  recall {cls:12s}: {r:.4f}")
    print()
    print(classification_report(targets, preds, labels=LABELS,
                                target_names=CANONICAL_CLASSES, digits=3, zero_division=0))

    cm = confusion_matrix(targets, preds, labels=LABELS)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CANONICAL_CLASSES)
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    disp.plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
    ax.set_title(f"Confusion Matrix - {name}")
    plt.tight_layout(); plt.show()

## 7. Run the evaluation on both unseen datasets

In [ ]:
for name, class_dir in [("BRISC 2025", OTHER1_CLASS_DIR),
                        ("Deep-Learning MRI", OTHER2_CLASS_DIR)]:
    loader = make_loader(class_dir)
    preds, targets = evaluate(model, loader, DEVICE)
    report(name, preds, targets)

## 8. (Optional) Sanity check on the original in-distribution test set

Runs the **same** loader/mapping/transform on the original Kaggle test set. If it reproduces the model's
in-training test accuracy, the pipeline is wired correctly and the external numbers can be trusted.
Only runs if the original dataset is present locally.

In [ ]:
ORIG_TEST_DIR = os.path.join("..", "prototype-models", "brain_mri_dataset", "Testing")
if os.path.isdir(ORIG_TEST_DIR):
    loader = make_loader(ORIG_TEST_DIR)
    preds, targets = evaluate(model, loader, DEVICE)
    report("Original test set (sanity check)", preds, targets)
else:
    print(f"Original test set not found at {ORIG_TEST_DIR} - skipping sanity check.")